# 25 — Ensemble RAG

> **Tier 6 | Ensemble & Meta**

## What is Ensemble RAG?

A single retrieval strategy has blind spots — a semantic search might miss an exact-term
match; a keyword search misses paraphrases. **Ensemble RAG** hedges by running multiple
strategies in parallel and merging their ranked results before generation.

### The three strategies

| Strategy | How it retrieves | Strength |
|----------|-----------------|----------|
| **Simple RAG** | Embed raw query → vector search | Fast, general-purpose |
| **HyDE** | LLM generates hypothetical answer → embed that → vector search | Bridges query-doc vocabulary gap |
| **Fusion RAG** | LLM generates N query variants → search each → RRF merge | Broader coverage via query diversity |

### Aggregation: Reciprocal Rank Fusion (RRF)

Each chunk's RRF score = `sum(1 / (k + rank_i))` over all result lists it appears in,
where `k=60` is a smoothing constant. Chunks retrieved by multiple strategies rise to
the top; chunks found by only one strategy still contribute.

```
rrf_score(chunk) = Σ  1 / (60 + rank_in_list_i)
                  i ∈ {simple, hyde, fusion}
```

## PDF Framework: pymupdf `get_text("text")`

This notebook uses **pymupdf** with `page.get_text("text")` — the plain-text mode.
Unlike `"dict"` (NB 19, span/font level) or `"blocks"` (NB 22, block objects),
`"text"` returns a single pre-formatted string per page: fastest pymupdf extraction,
no structural metadata needed.

| pymupdf mode | Returns | Used in |
|-------------|---------|--------|
| `get_text("dict")` | Span/line/font dicts | NB 19 — heading detection |
| `get_text("blocks")` | Block tuples + type flag | NB 22 — image skip, block_no |
| **`get_text("text")`** | **Plain string per page** | **NB 25 — fast plain extraction** |

## Flow Diagram


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pymupdf get_text(text)"]
        PDF(["📄 climate.pdf"])
        PDF --> MU["fitz.open()\npage.get_text('text')"]
        MU --> CH["Text chunks"]
        CH --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nensemble_rag_25")]
    end
    subgraph ENSEMBLE ["⚡  ENSEMBLE RETRIEVAL (parallel)"]
        Q(["❓ Query"])
        Q --> S1["Simple RAG\nembed raw query"]
        Q --> S2["HyDE\ngenerate hypothesis\n→ embed"]
        Q --> S3["Fusion RAG\ngenerate N variants\n→ search each"]
        S1 --> QDRANT
        S2 --> QDRANT
        S3 --> QDRANT
        QDRANT --> RRF["RRF Merge\n1/(k+rank)"]
        RRF --> TOP["Top-K merged chunks"]
        TOP --> GEN["LLM Generation\nClaude Sonnet 4.6"]
        GEN --> ANS(["✅ Final Answer"])
    end
    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style ENSEMBLE fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pymupdf", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid
from typing import List, Dict
from collections import defaultdict
from dotenv import load_dotenv

import boto3
import fitz   # pymupdf
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME = "ensemble_rag_25"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
STRATEGY_TOP_K  = 5    # chunks retrieved per strategy
FINAL_TOP_K     = 5    # top chunks after RRF merge passed to LLM
N_QUERY_VARIANTS = 3   # number of fusion query variants
RRF_K           = 60   # RRF smoothing constant

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection      : {COLLECTION_NAME}")
print(f"PDF             : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Strategies      : Simple + HyDE + Fusion ({N_QUERY_VARIANTS} variants)")
print(f"Chunks/strategy : {STRATEGY_TOP_K}  |  Final top-K : {FINAL_TOP_K}  |  RRF k={RRF_K}")


## Step 3 — Vector Store

In [ ]:
def make_qdrant(qdrant_url='', qdrant_api_key='', region='us-east-1'):
    if qdrant_url:
        try:
            kw = {'url': qdrant_url}
            if qdrant_api_key: kw['api_key'] = qdrant_api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {qdrant_url}')
            return client
        except Exception as e:
            print(f'Qdrant Cloud unavailable ({e}), falling back...')
    print('Using Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY, AWS_REGION)

cols = [c.name for c in qdrant.get_collections().collections]
if COLLECTION_NAME in cols:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
print(f'Created "{COLLECTION_NAME}" (dim={EMBEDDING_DIM})')


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

def call_llm(prompt: str, system: str = '', max_tokens: int = 512) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system or "You are a helpful assistant.",
        "messages": [{"role": "user", "content": prompt}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]

test_emb = embed_text("ensemble rag rrf pymupdf test")
print(f"Embedding OK — dim={len(test_emb)}")
print("call_llm ready.")


## Step 5 — PDF Loading with pymupdf `get_text("text")`

`page.get_text("text")` returns a single pre-formatted string per page — the fastest
pymupdf extraction mode, no structural metadata. Whitespace normalisation removes
excessive newlines left by the PDF renderer before chunking.


In [ ]:
def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def load_pdf_pymupdf_text(path: str):
    doc    = fitz.open(path)
    n_pages = len(doc)
    chunks  = []

    for page_num in range(n_pages):
        raw  = doc[page_num].get_text("text")
        # collapse runs of whitespace/newlines into single spaces
        text = ' '.join(raw.split()).strip()
        if not text:
            continue
        for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
            chunks.append({
                'text'      : sub,
                'page'      : page_num + 1,
                'char_count': len(sub),
            })

    doc.close()
    stats = {
        'n_pages' : n_pages,
        'n_chunks': len(chunks),
        'avg_chars': sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats


t0 = time.time()
chunks, stats = load_pdf_pymupdf_text(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pymupdf get_text('text') : {elapsed:.0f}ms")
print(f"Pages                    : {stats['n_pages']}")
print(f"Chunks                   : {stats['n_chunks']}")
print(f"Avg chunk length         : {stats['avg_chars']} chars")


## Step 6 — Index

In [ ]:
print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

pts = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=embs[i],
        payload={
            'text'    : chunks[i]['text'],
            'metadata': {
                'page'      : chunks[i]['page'],
                'char_count': chunks[i]['char_count'],
                'source'    : 'climate.pdf',
                'pdf_lib'   : 'pymupdf-text',
            }
        }
    )
    for i in range(len(chunks))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)
print(f"Indexed {qdrant.get_collection(COLLECTION_NAME).points_count} vectors in {time.time()-t0:.1f}s")


## Step 7 — The Three Retrieval Strategies

Each strategy returns a ranked list of `{text, metadata, score}` dicts.
Their results are fed into the RRF merger in Step 8.


In [ ]:
def vector_search(qvec: List[float], top_k: int = STRATEGY_TOP_K) -> List[Dict]:
    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME, query=qvec,
        limit=top_k, with_payload=True)
    return [{'text'    : p.payload['text'],
             'metadata': p.payload.get('metadata', {}),
             'score'   : p.score} for p in resp.points]


# --- Strategy 1: Simple RAG ---
def simple_rag_retrieve(question: str) -> List[Dict]:
    return vector_search(embed_text(question))


# --- Strategy 2: HyDE ---
def hyde_retrieve(question: str) -> List[Dict]:
    hypothesis = call_llm(
        f"Write a short factual passage (3-4 sentences) that directly answers: {question}",
        system="You are a technical writer. Write only the passage, no preamble.",
        max_tokens=200,
    )
    return vector_search(embed_text(hypothesis))


# --- Strategy 3: Fusion RAG ---
def fusion_retrieve(question: str, n: int = N_QUERY_VARIANTS) -> List[Dict]:
    variants_raw = call_llm(
        f"Generate {n} distinct search queries to retrieve information about: {question}\n"
        f"Output one query per line, no numbering or bullets.",
        system="You are a search query generator. Output only the queries.",
        max_tokens=200,
    )
    queries = [q.strip() for q in variants_raw.strip().splitlines() if q.strip()][:n]
    # search each variant, collect all ranked lists
    all_lists = [vector_search(embed_text(q)) for q in queries]
    # RRF merge across variant lists
    return rrf_merge(all_lists, top_k=STRATEGY_TOP_K)


print("Strategy functions defined: simple_rag_retrieve, hyde_retrieve, fusion_retrieve")


## Step 8 — RRF Merge

Reciprocal Rank Fusion score for each unique chunk across all result lists:

```
rrf(chunk) = Σ  1 / (k + rank)
```

Chunks appearing in multiple lists are boosted. The merged list is sorted
by descending RRF score and the top `FINAL_TOP_K` are kept.


In [ ]:
def rrf_merge(result_lists: List[List[Dict]], top_k: int = FINAL_TOP_K,
              k: int = RRF_K) -> List[Dict]:
    scores: Dict[str, float] = defaultdict(float)
    lookup: Dict[str, Dict]  = {}

    for result_list in result_lists:
        for rank, item in enumerate(result_list, start=1):
            key = item['text'][:120]   # dedup key
            scores[key] += 1.0 / (k + rank)
            if key not in lookup:
                lookup[key] = item

    merged = sorted(scores.items(), key=lambda x: -x[1])
    return [
        {**lookup[key], 'rrf_score': score}
        for key, score in merged[:top_k]
    ]


print("rrf_merge() defined.")


## Step 9 — Ensemble RAG Pipeline

All three strategies are executed sequentially (no threading needed for a notebook demo).
Their ranked lists are passed to `rrf_merge()`, the top chunks go to the LLM.


In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about a climate document.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""


def ensemble_rag(question: str, verbose: bool = True) -> Dict:
    t0 = time.time()

    # 1. Run all three strategies
    t_simple = time.time()
    simple_hits = simple_rag_retrieve(question)
    t_simple = (time.time() - t_simple) * 1000

    t_hyde = time.time()
    hyde_hits = hyde_retrieve(question)
    t_hyde = (time.time() - t_hyde) * 1000

    t_fusion = time.time()
    fusion_hits = fusion_retrieve(question)
    t_fusion = (time.time() - t_fusion) * 1000

    # 2. RRF merge across all three strategy outputs
    merged = rrf_merge([simple_hits, hyde_hits, fusion_hits], top_k=FINAL_TOP_K)

    # 3. Build context and generate
    context = '\n\n'.join(
        f"[{i+1}] page={h['metadata'].get('page','?')} rrf={h['rrf_score']:.4f}\n{h['text']}"
        for i, h in enumerate(merged)
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {question}"
    answer   = call_llm(user_msg, system=SYSTEM_PROMPT, max_tokens=1024)
    latency  = (time.time() - t0) * 1000

    if verbose:
        print(f"Q: {question}")
        print(f"Strategy latencies — Simple: {t_simple:.0f}ms | HyDE: {t_hyde:.0f}ms | Fusion: {t_fusion:.0f}ms")
        print(f"Unique chunks after RRF: {len(merged)}  (from {len(simple_hits)+len(hyde_hits)+len(fusion_hits)} total)")
        print(f"A: {answer[:600]}")
        print(f"   [total {latency:.0f}ms]")
        print()

    return {
        'question'    : question,
        'answer'      : answer,
        'simple_hits' : simple_hits,
        'hyde_hits'   : hyde_hits,
        'fusion_hits' : fusion_hits,
        'merged'      : merged,
        'latency_ms'  : latency,
        't_simple_ms' : t_simple,
        't_hyde_ms'   : t_hyde,
        't_fusion_ms' : t_fusion,
    }


print("ensemble_rag() defined.")


## Step 10 — Demo

In [ ]:
questions = [
    "How does Numerical Weather Prediction work?",
    "What satellite systems are used to collect weather data?",
    "What are the main challenges in long-range weather forecasting?",
]

results = []
print("=" * 70)
for q in questions:
    r = ensemble_rag(q, verbose=True)
    results.append(r)
    print("-" * 70)


## Step 11 — RRF Overlap Analysis

Show how many chunks were found by 1, 2, or all 3 strategies per question —
chunks found by multiple strategies have higher RRF scores.


In [ ]:
print(f"{'Strategy':<12} {'Avg latency':>12}")
print("-" * 26)
print(f"{'Simple':<12} {sum(r['t_simple_ms'] for r in results)/len(results):>11.0f}ms")
print(f"{'HyDE':<12} {sum(r['t_hyde_ms'] for r in results)/len(results):>11.0f}ms")
print(f"{'Fusion':<12} {sum(r['t_fusion_ms'] for r in results)/len(results):>11.0f}ms")
print(f"{'Total':<12} {sum(r['latency_ms'] for r in results)/len(results):>11.0f}ms")
print()

for r in results:
    simple_keys  = {h['text'][:120] for h in r['simple_hits']}
    hyde_keys    = {h['text'][:120] for h in r['hyde_hits']}
    fusion_keys  = {h['text'][:120] for h in r['fusion_hits']}

    only_one  = len((simple_keys | hyde_keys | fusion_keys) -
                    ((simple_keys & hyde_keys) | (simple_keys & fusion_keys) | (hyde_keys & fusion_keys)))
    two_plus  = len((simple_keys & hyde_keys) | (simple_keys & fusion_keys) | (hyde_keys & fusion_keys))
    all_three = len(simple_keys & hyde_keys & fusion_keys)

    print(f"Q: {r['question'][:60]}")
    print(f"   Found by 1 strategy only : {only_one}")
    print(f"   Found by 2+ strategies   : {two_plus}")
    print(f"   Found by all 3           : {all_three}")
    print(f"   Top RRF chunk score      : {r['merged'][0]['rrf_score']:.4f}")
    print()


## Step 12 — Summary

### Ensemble RAG at a glance

| Dimension | Single-strategy RAG | Ensemble RAG |
|-----------|--------------------|--------------|
| Retrieval coverage | One angle | Three complementary angles |
| Vocabulary gap | Only if semantic | HyDE bridges query-doc gap |
| Query diversity | Fixed | Fusion generates N variants |
| Aggregation | N/A | RRF — no score calibration needed |
| Latency | Low | Higher (3× LLM + embed calls) |

### RRF scoring intuition

```
A chunk ranked #1 by all 3 strategies:
  rrf = 1/(60+1) + 1/(60+1) + 1/(60+1) = 0.0492

A chunk ranked #1 by only 1 strategy:
  rrf = 1/(60+1) = 0.0164

Cross-strategy agreement triples the score.
```

### pymupdf mode comparison

| Mode | Output | Use case |
|------|--------|----------|
| `get_text("text")` | Plain string | Fast plain extraction (this notebook) |
| `get_text("blocks")` | Block tuples + type | Image skip, block_no metadata |
| `get_text("dict")` | Nested span/font dicts | Heading detection via font size |

> **Next: [26 — Adaptive RAG](26_Adaptive_RAG.ipynb)** —
> route each query to the best single strategy via a classifier, avoiding the latency cost
> of always running all three.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {qdrant.get_collection(COLLECTION_NAME).points_count} vectors")
print(f"PDF framework  : pymupdf get_text('text') — plain-text mode")
print(f"RAG pattern    : Ensemble RAG — Simple + HyDE + Fusion, merged with RRF")
print(f"Strategies     : {N_QUERY_VARIANTS} fusion variants, RRF k={RRF_K}, final top-{FINAL_TOP_K}")
print()
print("Notebook 25 — Ensemble RAG complete.")
